# 코랩으로 만드는 AI 에이전트 챗봇
**색소물이 튜브를 통과하는 시간을 맞춰 볼까요? · 60분**

AI 에이전트는 질문을 보고 **어떤 도구를 쓸지 정하고**, 도구에서 나온 값으로 다음 행동을 선택할 수 있어요. 이 실습에서는 매뉴얼을 찾고, 유량을 계산하고, 펌프가 그 유량을 낼 수 있는지 확인합니다.

**시작하기** · 이 파일을 Google Colab에 올리고 **런타임 → 모두 실행**을 누르세요. 기본 설정은 모의 모드입니다. API 키 없이 계산과 챗봇을 실행할 수 있어요. PubChem을 조회한 뒤, API 키가 있다면 Gemini를 연결해 본 실습을 진행하세요.

| 시간 | 해볼 일 |
|---|---|
| 0–10분 | 수업용 매뉴얼에서 펌프 한계와 튜브 부피 읽기 |
| 10–20분 | PubChem 공개 API에 물의 분자식과 몰 질량을 물어보기 |
| 20–35분 | 기존 Gemini 키를 연결하고 챗봇에 질문해 도구 호출 확인하기 |
| 35–55분 | 목표 시간과 질문을 바꾸며 다섯 문제 풀기 |
| 55–60분 | 두 모드의 호출 기록 비교하기 |

튜브와 펌프 조건은 수업용 가상 데이터입니다. 화면에서 설정 후보를 계산해 제안하는 실습이에요.


In [ ]:
# @title 준비 1 · 매뉴얼, 계산기, PubChem 도구
import json
import math
import re
from urllib.error import URLError
from urllib.request import Request, urlopen

# 두 문서는 수업을 위해 만든 가상 장비 자료입니다.
PUMP = {"source": "PUMP-01", "max_flow_ml_min": 1.5}
TUBES = {"source": "TUBE-01", "tube_volume_ml": {"A": 4.0, "B": 2.0}}
PUBCHEM_URL = (
    "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/962/"
    "property/MolecularFormula,MolecularWeight/JSON"
)
TOOL_LABELS = {
    "find_manual": "매뉴얼 찾기",
    "calculate_flow": "필요한 유량 계산",
    "check_pump_limit": "펌프 한계 확인",
    "lookup_water": "PubChem 물 정보 조회",
}


def find_manual(topic):
    if topic == "pump":
        return dict(PUMP)
    if topic == "tubes":
        return {"source": TUBES["source"], "tube_volume_ml": dict(TUBES["tube_volume_ml"])}
    return {"error": "topic은 pump 또는 tubes 중 하나여야 합니다."}


def positive_number(value):
    if type(value) not in (int, float):
        return False
    try:
        return math.isfinite(value) and value > 0
    except OverflowError:
        return False


def calculate_flow(tube_id, target_minutes):
    if tube_id not in TUBES["tube_volume_ml"]:
        return {"error": "튜브는 A 또는 B만 사용할 수 있습니다."}
    if not positive_number(target_minutes):
        return {"error": "목표 시간을 0보다 큰 분 단위 숫자로 넣으세요."}
    volume = TUBES["tube_volume_ml"][tube_id]
    flow = volume / target_minutes
    if not math.isfinite(flow):
        return {"error": "목표 시간이 너무 작아서 유량을 계산할 수 없습니다."}
    return {
        "tube_id": tube_id,
        "volume_ml": volume,
        "target_minutes": target_minutes,
        "flow_ml_min": flow,
        "source": TUBES["source"],
    }


def check_pump_limit(flow_ml_min):
    if not positive_number(flow_ml_min):
        return {"error": "유량을 0보다 큰 mL/min 단위 숫자로 넣으세요."}
    return {
        "flow_ml_min": flow_ml_min,
        "max_flow_ml_min": PUMP["max_flow_ml_min"],
        "allowed": flow_ml_min <= PUMP["max_flow_ml_min"],
        "source": PUMP["source"],
    }


def lookup_water():
    """키 없이 PubChem CID 962의 분자식과 분자량을 읽습니다."""
    try:
        request = Request(PUBCHEM_URL, headers={"Accept": "application/json"})
        with urlopen(request, timeout=12) as response:
            data = json.load(response)
        row = data["PropertyTable"]["Properties"][0]
        return {
            "cid": row["CID"],
            "formula": row["MolecularFormula"],
            "molar_mass_g_mol": float(row["MolecularWeight"]),
            "source": "https://pubchem.ncbi.nlm.nih.gov/compound/962",
        }
    except (URLError, TimeoutError, OSError, ValueError, KeyError, IndexError, TypeError):
        return {"error": "PubChem 응답을 가져오지 못했습니다. 잠시 뒤 다시 시도해 보세요."}


# 모델은 아래에 등록된 네 도구만 요청할 수 있습니다.
TOOL_DECLARATIONS = [
    {
        "name": "find_manual",
        "description": "수업용 가상 매뉴얼 조회. pump는 펌프 최대 유량, tubes는 A/B 튜브 부피.",
        "parameters": {"type": "OBJECT", "properties": {
            "topic": {"type": "STRING", "enum": ["pump", "tubes"]},
        }, "required": ["topic"]},
    },
    {
        "name": "calculate_flow",
        "description": "튜브 A/B의 유량을 부피(mL) ÷ 목표 시간(분)으로 계산합니다.",
        "parameters": {"type": "OBJECT", "properties": {
            "tube_id": {"type": "STRING", "enum": ["A", "B"]},
            "target_minutes": {"type": "NUMBER", "description": "0보다 큰 시간, 단위는 분"},
        }, "required": ["tube_id", "target_minutes"]},
    },
    {
        "name": "check_pump_limit",
        "description": "계산한 유량을 가상 펌프 한계와 비교해 allowed를 반환합니다.",
        "parameters": {"type": "OBJECT", "properties": {
            "flow_ml_min": {"type": "NUMBER", "description": "필요 유량, 단위는 mL/min"},
        }, "required": ["flow_ml_min"]},
    },
    {
        "name": "lookup_water",
        "description": "PubChem 공개 API에서 물(CID 962)의 분자식과 분자량을 조회합니다.",
    },
]


def execute_tool(name, args):
    if not isinstance(args, dict):
        return {"error": "도구에 전달한 값은 이름과 값으로 된 객체여야 합니다."}
    fields = {
        "find_manual": {"topic"},
        "calculate_flow": {"tube_id", "target_minutes"},
        "check_pump_limit": {"flow_ml_min"},
        "lookup_water": set(),
    }
    if not isinstance(name, str) or name not in fields:
        return {"error": "수업에 등록된 도구만 사용할 수 있습니다."}
    if set(args) != fields[name]:
        return {"error": "이 도구에 필요한 값: " + (", ".join(sorted(fields[name])) or "없음")}
    if name == "find_manual":
        return find_manual(args["topic"])
    if name == "calculate_flow":
        if not isinstance(args["tube_id"], str):
            return {"error": "튜브 이름은 A 또는 B입니다."}
        return calculate_flow(args["tube_id"], args["target_minutes"])
    if name == "check_pump_limit":
        return check_pump_limit(args["flow_ml_min"])
    return lookup_water()


def call_and_record(trace, name, **args):
    result = execute_tool(name, args)
    trace.append({"name": name, "args": args, "result": result})
    return result


def demo_plan(target_minutes):
    """수업용 규칙으로 도구를 차례대로 실행합니다."""
    trace = []
    if not positive_number(target_minutes):
        return {"reply": "목표 통과 시간을 알려 주세요. 예: 2분 또는 90초.", "trace": trace}
    pump = call_and_record(trace, "find_manual", topic="pump")
    tubes = call_and_record(trace, "find_manual", topic="tubes")
    observations = []
    allowed = []
    for tube_id in tubes["tube_volume_ml"]:
        calc = call_and_record(trace, "calculate_flow", tube_id=tube_id, target_minutes=target_minutes)
        if "error" in calc:
            return {"reply": calc["error"], "trace": trace}
        flow = calc["flow_ml_min"]
        check = call_and_record(trace, "check_pump_limit", flow_ml_min=flow)
        status = "가능" if check["allowed"] else "한계 초과"
        observations.append(f"튜브 {tube_id}: {flow:.4g} mL/min ({status})")
        if check["allowed"]:
            allowed.append(tube_id)
    summary = f"목표 {target_minutes:.4g}분 · 펌프 한계 {pump['max_flow_ml_min']:g} mL/min\n"
    summary += "\n".join(observations) + "\n"
    if allowed:
        summary += "사용 가능: " + ", ".join(f"튜브 {tube}" for tube in allowed) + "."
    else:
        summary += "현재 튜브와 펌프로는 가능한 조합이 없습니다. 목표 시간을 늘려 보세요."
    return {"reply": summary, "trace": trace}


def demo_chat(question):
    question = question.strip()
    if any(token in question for token in ("분자식", "분자량", "몰 질량", "PubChem", "pubchem")):
        trace = []
        row = call_and_record(trace, "lookup_water")
        answer = row.get("error") or (
            f"물의 분자식은 {row['formula']}, 몰 질량은 {row['molar_mass_g_mol']:g} g/mol입니다.\n"
            f"출처: {row['source']}"
        )
        return {"reply": answer, "trace": trace}
    matches = re.findall(r"([+-]?(?:\d+(?:\.\d+)?|\.\d+))\s*(분|초)", question)
    if len(matches) == 1:
        number, unit = matches[0]
        return demo_plan(float(number) / (60 if unit == "초" else 1))
    if not matches and "유량" in question and any(x in question for x in ("최대", "한계")):
        trace = []
        pump = call_and_record(trace, "find_manual", topic="pump")
        return {"reply": f"펌프 최대 유량은 {pump['max_flow_ml_min']:g} mL/min입니다. [{pump['source']}]",
                "trace": trace}
    return {"reply": "목표 통과 시간을 하나 알려 주세요. 예: 색소물이 2분에 통과하게 해 줘.", "trace": []}


def show_result(result):
    print(result["reply"])
    if result["trace"]:
        print("\n도구 호출 기록")
        for step, call in enumerate(result["trace"], 1):
            print(f"{step}. {TOOL_LABELS.get(call['name'], call['name'])} {call['args']}")
            print(f"   → {call['result']}")
    else:
        print("\n도구 호출 없음")

print("준비 완료. 매뉴얼, 계산기, 공개 데이터 조회 도구를 불러왔습니다.")


In [ ]:
# @title 준비 2 · Gemini가 도구를 요청할 때 실행하는 코드
import json
import re
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


GEMINI_MODEL = "gemini-3.5-flash-lite"
MAX_GEMINI_CALLS = 5
GEMINI_INSTRUCTION = """당신은 물과 색소를 사용하는 모의 실험의 한국어 조교입니다.
질문에 답할 때 장비 조건이나 계산값은 등록된 도구로 확인하세요.
- 장비 규칙이나 튜브 부피는 find_manual로 찾으세요.
- 목표 통과 시간이 없거나 양수가 아니면 올바른 시간을 물어보세요. 초로 받은 시간은 분으로 바꾸세요.
- 통과 시간을 맞추는 요청이면 펌프 한계를 찾고, 후보 튜브의 유량을 calculate_flow로 계산한 뒤 check_pump_limit으로 판정하세요.
- 기본 후보는 매뉴얼의 A와 B입니다. 두 매뉴얼 조회는 한 응답에 묶으세요. A/B 계산도 한 응답에 묶고, 두 한계 판정도 한 응답에 묶으세요.
- 가능한 후보와 불가능한 이유를 알려 주세요. 모두 불가능하면 가능한 후보가 없다고 답하세요.
- 물의 분자식이나 분자량을 질문하면 lookup_water로 공개 자료를 확인하세요.
- 도구 결과의 수치, 단위와 근거를 짧게 설명하세요. 도구가 실패하면 결과를 지어내지 마세요.
실제 장비를 조작한다는 말은 하지 마세요. 질문이 모의 실험 범위를 벗어나면 이 실습에서 확인할 수 있는 항목을 안내하세요."""


def _gemini_request_json(payload, api_key, model):
    """공식 Gemini API에 요청하고 JSON 응답을 받습니다."""
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"
    request = Request(
        url,
        data=json.dumps(payload, ensure_ascii=False).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    # 다른 주소로 리디렉션되더라도 API 키 헤더를 전달하지 않습니다.
    request.add_unredirected_header("x-goog-api-key", api_key)
    try:
        with urlopen(request, timeout=20) as response:
            return json.load(response)
    except HTTPError as error:
        messages = {
            400: "Gemini가 요청을 처리하지 못했습니다. 모델 이름과 요청 설정을 확인하세요.",
            401: "Gemini API 키를 확인하세요.",
            403: "Gemini 이용 권한이나 프로젝트의 API 설정을 확인하세요.",
            404: "이 프로젝트에서 선택한 Gemini 모델을 찾을 수 없습니다.",
            429: "Gemini 사용 한도에 도달했습니다. 잠시 뒤 다시 시도하거나 연습 모드를 사용하세요.",
        }
        message = messages.get(error.code, f"Gemini API 요청이 실패했습니다 (HTTP {error.code}).")
        raise RuntimeError(message) from None
    except (URLError, OSError):
        raise RuntimeError("Gemini API에 연결하지 못했습니다. 코랩의 연결 상태를 확인하세요.") from None
    except ValueError:
        raise RuntimeError("Gemini의 응답을 읽지 못했습니다. 잠시 뒤 다시 시도하세요.") from None


def gemini_turn(question, api_key, history=None, model=GEMINI_MODEL, request_json=None):
    """한 질문에 답하고, 이 질문에서 실행한 도구와 대화 기록을 돌려줍니다."""
    if not isinstance(question, str) or not question.strip():
        raise RuntimeError("질문을 먼저 입력하세요.")
    if not isinstance(api_key, str) or not api_key.strip() or "\n" in api_key or "\r" in api_key:
        raise RuntimeError("코랩 보안 비밀에 GEMINI_API_KEY를 설정하세요.")
    if not isinstance(model, str) or not re.fullmatch(r"gemini-[a-z0-9][a-z0-9.-]{0,79}", model):
        raise RuntimeError("Gemini 모델 이름을 확인하세요.")

    send = request_json if request_json is not None else _gemini_request_json
    previous = list(history or [])
    contents = previous + [{"role": "user", "parts": [{"text": question.strip()}]}]
    allowed_tools = {tool["name"] for tool in TOOL_DECLARATIONS}
    trace = []

    for turn in range(MAX_GEMINI_CALLS):
        payload = {
            "systemInstruction": {"parts": [{"text": GEMINI_INSTRUCTION}]},
            "contents": contents,
            "tools": [{"functionDeclarations": TOOL_DECLARATIONS}],
            "toolConfig": {"functionCallingConfig": {"mode": "AUTO"}},
            "generationConfig": {"maxOutputTokens": 2048},
        }
        response = send(payload, api_key.strip(), model)
        candidates = response.get("candidates", []) if isinstance(response, dict) else []
        if not isinstance(candidates, list) or not candidates or not isinstance(candidates[0], dict):
            raise RuntimeError("Gemini에서 답변을 받지 못했습니다. 질문을 바꿔 다시 시도하세요.")
        candidate = candidates[0]
        content = candidate.get("content") or {}
        parts = content.get("parts", []) if isinstance(content, dict) else []
        if candidate.get("finishReason") == "MAX_TOKENS":
            raise RuntimeError("Gemini 응답 길이 한도에 도달했습니다. 질문을 하나씩 나눠 주세요.")
        if not isinstance(parts, list) or not parts or not all(isinstance(part, dict) for part in parts):
            raise RuntimeError("Gemini에서 표시할 답변을 받지 못했습니다. 질문을 바꿔 보세요.")

        # 모델이 보낸 원래 메시지와 Parts를 그대로 돌려줍니다.
        # 이 안의 thoughtSignature는 Gemini가 호출의 맥락을 이어 가는 데 씁니다.
        if not content.get("role"):
            content = {**content, "role": "model"}
        contents = contents + [content]
        calls = [part["functionCall"] for part in parts if "functionCall" in part]
        if not calls:
            reply = "\n".join(
                part["text"] for part in parts
                if not part.get("thought") and isinstance(part.get("text"), str)
            ).strip()
            if not reply:
                raise RuntimeError("Gemini에서 표시할 답변을 받지 못했습니다. 질문을 바꿔 보세요.")
            return {"reply": reply, "trace": trace, "history": contents}

        if turn == MAX_GEMINI_CALLS - 1:
            break

        # 이번 응답에서 요청한 등록 함수만 실행하고, 호출 ID를 함께 돌려줍니다.
        if len(calls) > 8:
            raise RuntimeError("Gemini가 한 번에 너무 많은 도구를 요청했습니다. 질문을 나눠 주세요.")
        result_parts = []
        for call in calls:
            if not isinstance(call, dict):
                raise RuntimeError("Gemini의 도구 요청 형식을 읽지 못했습니다.")
            name = call.get("name")
            args = call.get("args", {})
            if not isinstance(name, str) or name not in allowed_tools or not isinstance(args, dict):
                raise RuntimeError("Gemini가 등록된 도구나 입력 형식을 벗어난 호출을 요청했습니다.")
            result = execute_tool(name, dict(args))
            trace.append({"name": name, "args": args, "result": result})
            function_response = {"name": name, "response": result}
            if call.get("id") is not None:
                function_response["id"] = call["id"]
            result_parts.append({"functionResponse": function_response})
        contents = contents + [{"role": "user", "parts": result_parts}]

    return {
        "reply": "이번 질문의 API 호출 한도(5회)에 도달했습니다. 질문을 한 가지씩 나눠 다시 시도해 주세요.",
        "trace": trace,
        "history": previous,
    }


## 1. 매뉴얼 읽고 먼저 예상하기

> 색소물이 튜브를 통과하는 데 **2분**이 걸리게 하려면?

| 가상 매뉴얼 | 내용 |
|---|---|
| PUMP-01 | 펌프는 분당 최대 **1.5 mL**까지 보낼 수 있어요. |
| TUBE-01 | 튜브 A에 물이 **4 mL**, 튜브 B에 물이 **2 mL** 들어 있어요. |

**필요한 유량 = 튜브 안의 물의 양 ÷ 목표 시간**이에요. 이 단순 모형에서는 튜브가 물로 가득 차 있고 유량이 일정하다고 가정합니다.

아래 셀을 실행하고 A와 B의 결과를 보세요. 한계를 넘긴 후보는 어떤 도구에서 걸러졌나요?


In [ ]:
target_minutes = 2
show_result(demo_plan(target_minutes))


## 2. 공개 데이터 API에 직접 물어보기

[PubChem](https://pubchem.ncbi.nlm.nih.gov/compound/962)은 물질 정보를 제공하는 공개 데이터베이스예요. 아래 체크박스를 켜고 실행하면 물의 분자식과 몰 질량을 가져옵니다. API 키가 필요 없습니다.

Python은 PubChem에 물의 번호 **962**를 보내고, 돌려받은 JSON에서 두 항목만 골라 보여줘요. 연결이 어려우면 [PubChem 물 페이지](https://pubchem.ncbi.nlm.nih.gov/compound/962)에서 확인하세요.


In [ ]:
# @title PubChem 실제 조회
fetch_pubchem = False  # @param {type:"boolean"}

if fetch_pubchem:
    show_result(demo_chat("물의 분자식과 몰 질량을 PubChem에서 찾아 줘."))
else:
    print("fetch_pubchem에 체크하고 이 셀을 실행하면 PubChem에 접속합니다.")


## 3. Gemini 연결하기 · 성인 수강생 선택

**모의 모드**에서는 수업용 Python 규칙이 도구를 고릅니다. **Gemini 모드**에서는 실제 언어모델이 도구를 요청합니다. Python은 요청받은 계산이나 조회를 실행하고, 그 결과를 모델에 돌려줍니다.

1. 이미 발급받은 Gemini API 키를 준비하세요. 새 키가 필요한 경우에만 [Google AI Studio](https://aistudio.google.com/apikey)에서 발급받으세요.
2. 코랩 왼쪽 **보안 비밀(열쇠 아이콘)**에서 기존 키를 확인하세요. 처음 저장한다면 **GEMINI_API_KEY** 이름으로 추가하세요. 이 노트북의 액세스를 켜세요.
3. 아래에서 저장한 비밀 이름을 고르고 **enable_gemini**를 체크해 실행하세요. 다음 챗봇 셀에서 **Gemini** 모드로 질문하면 됩니다.

키는 보안 비밀에만 저장하세요. 모델로 보내는 질문에는 수업 예시를 사용하세요. [Gemini API는 만 18세 이상 사용](https://ai.google.dev/gemini-api/terms?hl=ko)을 조건으로 합니다. 무료 등급은 [공식 안내](https://ai.google.dev/gemini-api/docs/pricing?hl=ko)에서 확인하세요.


In [ ]:
# @title Gemini 연결
secret_name = "GEMINI_API_KEY"  # @param ["GEMINI_API_KEY", "GOOGLE_API_KEY"]
enable_gemini = False  # @param {type:"boolean"}
gemini_key = None

if enable_gemini:
    try:
        from google.colab import userdata
        gemini_key = userdata.get(secret_name)
        if not isinstance(gemini_key, str) or not gemini_key.strip():
            gemini_key = None
            print("코랩 보안 비밀에 선택한 이름으로 API 키를 넣어 주세요.")
        else:
            print("키를 불러왔습니다. 챗봇에서 Gemini 모드를 선택해 질문하세요.")
    except ImportError:
        print("이 연결 셀은 코랩에서 실행하세요. 기본 실습은 모의 모드로 진행할 수 있습니다.")
    except Exception:
        print("선택한 보안 비밀 이름과 이 노트북의 액세스를 확인하세요.")
else:
    print("기본 설정은 모의 모드입니다. Gemini를 쓰려면 enable_gemini를 켜고 다시 실행하세요.")


## 4. 챗봇에 질문하기

아래 셀을 실행하면 **질문 입력창**이 열립니다. 모드를 고르고 **보내기**를 눌러 보세요. 답 아래에 호출한 도구와 결과가 표시됩니다.

Gemini 모드에서는 “그럼 90초라면?”처럼 이어서 질문해 보세요. 새 상황을 시험할 때는 **기록 지우기**를 누르세요. 질문 하나에 Gemini API를 최대 5회 호출합니다.


In [ ]:
# @title 챗봇 화면 열기
gemini_turns = []


def ask(question, mode="모의"):
    """입력창의 보내기 버튼을 누르거나, ask(...)로 직접 질문합니다."""
    question = question.strip()
    if not question or len(question) > 500:
        return {"reply": "질문을 1~500자로 입력해 주세요.", "trace": []}
    if mode == "모의":
        return demo_chat(question)
    if mode != "Gemini":
        return {"reply": "모드 이름은 '모의' 또는 'Gemini'입니다.", "trace": []}
    if not gemini_key:
        return {"reply": "위의 ‘Gemini 연결’ 셀에서 키를 불러오세요.", "trace": []}

    previous = [message for turn in gemini_turns[-3:] for message in turn]
    try:
        result = gemini_turn(question, gemini_key, history=previous)
    except RuntimeError as error:
        return {"reply": str(error), "trace": []}
    new_turn = result["history"][len(previous):]
    if new_turn:
        gemini_turns.append(new_turn)
        del gemini_turns[:-3]
    return {"reply": result["reply"], "trace": result["trace"]}


try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    print("이 환경에서는 입력창을 열 수 없습니다. 아래 ask(...) 예시 셀을 실행하세요.")
else:
    prompt = widgets.Text(value="색소물이 2분에 통과하게 해 줘.", description="질문",
                          layout=widgets.Layout(width="100%"))
    mode_box = widgets.Dropdown(options=["모의", "Gemini"], value="Gemini" if gemini_key else "모의", description="모드")
    send_button = widgets.Button(description="보내기", button_style="primary")
    reset_button = widgets.Button(description="기록 지우기")
    output = widgets.Output(layout=widgets.Layout(max_height="480px", overflow="auto"))
    status = widgets.Label(value="질문을 적고 보내기를 누르세요.")

    def send(_):
        question = prompt.value.strip()
        if not question:
            status.value = "질문을 입력하세요."
            return
        send_button.disabled = True
        reset_button.disabled = True
        status.value = "답변을 확인하는 중입니다."
        try:
            result = ask(question, mode=mode_box.value)
            with output:
                print(f"\n나: {question}\n챗봇 ({mode_box.value}):")
                show_result(result)
        finally:
            send_button.disabled = False
            reset_button.disabled = False
            status.value = "다음 질문을 입력할 수 있습니다."

    def reset(_):
        gemini_turns.clear()
        output.clear_output()
        status.value = "대화 기록을 지웠습니다."

    send_button.on_click(send)
    reset_button.on_click(reset)
    display(widgets.VBox([prompt, widgets.HBox([mode_box, send_button, reset_button]), status, output]))


입력창이 보이지 않으면 아래 문장을 고쳐 실행해도 됩니다. 키 연결을 완료했다면 모드에 **Gemini**를 넣을 수 있어요.


In [ ]:
question = "색소물이 2분에 통과하게 해 줘."
show_result(ask(question, mode="모의"))


## 5. 실습 문제

| 문제 | 챗봇에 넣을 문장 | 확인할 점 |
|---|---|---|
| 1. 조회만 하기 | 펌프의 최대 유량이 얼마야? | 펌프 매뉴얼과 근거 번호가 나왔나요? |
| 2. 계산하고 비교하기 | 색소물이 2분에 통과하게 해 줘. | A와 B를 계산하고 한계를 비교했나요? |
| 3. 조건이 부족할 때 | 색소물이 잘 통과하게 해 줘. | 기록을 지운 뒤 질문해요. 먼저 시간을 되물었나요? |
| 4. 가능한 후보가 없을 때 | 색소물이 0.5분에 통과하게 해 줘. | 두 후보 모두 안 된다고 설명했나요? |
| 5. 공개 데이터 읽기 | 물의 분자식과 몰 질량을 PubChem에서 찾아 줘. | 조회 도구, 출처, 단위가 나왔나요? |

두 모드에 **같은 질문**을 넣어 보세요. 어떤 도구를 썼나요? 계산값과 단위는 맞았나요? 정보가 빠졌거나 한계를 넘으면 어떻게 답했나요?


In [ ]:
# @title 예시 답안 펼치기
show_answers = False  # @param {type:"boolean"}

if show_answers:
    print('''1. 펌프 최대 유량 1.5 mL/min (PUMP-01). 도구: 매뉴얼 찾기.
2. A는 4 ÷ 2 = 2 mL/min이라 한계 초과. B는 2 ÷ 2 = 1 mL/min이라 가능.
3. 새 대화에서는 목표 통과 시간을 먼저 질문.
4. A는 8 mL/min, B는 4 mL/min. 현재 장비로는 둘 다 불가능.
5. H2O, 18.015 g/mol. 출처 PubChem CID 962. 조회가 실패하면 실패를 알림.''')
else:
    print("문제를 풀고 show_answers를 켜면 답을 확인할 수 있습니다.")


## 세부 코드 보기 · 선택

먼저 준비 셀을 실행하세요. 주제를 고르고 **show_code**를 켜면 실행에 사용한 소스가 셀 아래에 나옵니다. 원본 코드는 각 준비 셀과 챗봇 셀에서도 펼쳐 볼 수 있어요.

공식 문서 · [Gemini 함수 호출](https://ai.google.dev/gemini-api/docs/generate-content/function-calling) · [PubChem PUG REST](https://pubchem.ncbi.nlm.nih.gov/docs/pug-rest)


In [ ]:
# @title 세부 코드 펼치기
topic = "조회와 계산"  # @param ["조회와 계산", "Gemini 도구 호출", "챗봇"]
show_code = False  # @param {type:"boolean"}

if show_code:
    import linecache
    from IPython.display import Code, display
    function_name = {
        "조회와 계산": "demo_plan",
        "Gemini 도구 호출": "gemini_turn",
        "챗봇": "ask",
    }.get(topic)
    fn = globals().get(function_name)
    if fn is None:
        print("해당 준비 셀 또는 챗봇 셀을 실행한 뒤 다시 시도해 주세요.")
    else:
        source = "".join(linecache.getlines(fn.__code__.co_filename))
        if source:
            display(Code(data=source, language="python"))
        else:
            print("해당 셀에서 원본 코드를 펼쳐 확인하세요.")
else:
    print("세부 코드를 보고 싶으면 주제를 고르고 show_code를 켜세요.")
